# OIS Forward Curves

Loads OIS swap-rate node data from Bloomberg and bootstraps a full 360-month
forward-rate curve for EUR, USD, GBP and JPY using a PCHIP cubic spline on the
cumulative discount function A(t) = −log P(t).

**Requires** a live Bloomberg Terminal and `pxts[bloomberg]` installed:
```
pip install "pxts[bloomberg]"
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
import plotly.graph_objects as go

import pxts
from pxts import read_bdh, FT_COLORS

## Ticker reference table

OIS node tickers (Bloomberg BGN) and their maturities in years.

In [ ]:
curve_tix = pd.DataFrame({
    "EUR": [
        "EESWEA BGN Curncy", "EESWEB BGN Curncy", "EESWEC BGN Curncy", "EESWED BGN Curncy",
        "EESWEE BGN Curncy", "EESWEF BGN Curncy", "EESWEG BGN Curncy", "EESWEH BGN Curncy",
        "EESWEI BGN Curncy", "EESWEJ BGN Curncy", "EESWEK BGN Curncy", "EESWE1 BGN Curncy",
        "EESWE1F BGN Curncy", "EESWE2 BGN Curncy", "EESWE3 BGN Curncy", "EESWE4 BGN Curncy",
        "EESWE5 BGN Curncy", "EESWE6 BGN Curncy", "EESWE7 BGN Curncy", "EESWE8 BGN Curncy",
        "EESWE9 BGN Curncy", "EESWE10 BGN Curncy", "EESWE12 BGN Curncy", "EESWE15 BGN Curncy",
        "EESWE20 BGN Curncy", "EESWE25 BGN Curncy", "EESWE30 BGN Curncy",
    ],
    "USD": [
        "USOSFRA BGN Curncy", "USOSFRB BGN Curncy", "USOSFRC BGN Curncy", "USOSFRD BGN Curncy",
        "USOSFRE BGN Curncy", "USOSFRF BGN Curncy", "USOSFRG BGN Curncy", "USOSFRH BGN Curncy",
        "USOSFRI BGN Curncy", "USOSFRJ BGN Curncy", "USOSFRK BGN Curncy", "USOSFR1 BGN Curncy",
        "USOSFR1F BGN Curncy", "USOSFR2 BGN Curncy", "USOSFR3 BGN Curncy", "USOSFR4 BGN Curncy",
        "USOSFR5 BGN Curncy", "USOSFR6 BGN Curncy", "USOSFR7 BGN Curncy", "USOSFR8 BGN Curncy",
        "USOSFR9 BGN Curncy", "USOSFR10 BGN Curncy", "USOSFR12 BGN Curncy", "USOSFR15 BGN Curncy",
        "USOSFR20 BGN Curncy", "USOSFR25 BGN Curncy", "USOSFR30 BGN Curncy",
    ],
    "GBP": [
        "BPSWSA BGN Curncy", "BPSWSB BGN Curncy", "BPSWSC BGN Curncy", "BPSWSD BGN Curncy",
        "BPSWSE BGN Curncy", "BPSWSF BGN Curncy", "BPSWSG BGN Curncy", "BPSWSH BGN Curncy",
        "BPSWSI BGN Curncy", "BPSWSJ BGN Curncy", "BPSWSK BGN Curncy", "BPSWS1 BGN Curncy",
        "BPSWS1F BGN Curncy", "BPSWS2 BGN Curncy", "BPSWS3 BGN Curncy", "BPSWS4 BGN Curncy",
        "BPSWS5 BGN Curncy", "BPSWS6 BGN Curncy", "BPSWS7 BGN Curncy", "BPSWS8 BGN Curncy",
        "BPSWS9 BGN Curncy", "BPSWS10 BGN Curncy", "BPSWS12 BGN Curncy", "BPSWS15 BGN Curncy",
        "BPSWS20 BGN Curncy", "BPSWS25 BGN Curncy", "BPSWS30 BGN Curncy",
    ],
    "JPY": [
        "JYSOA BGN Curncy", "JYSOB BGN Curncy", "JYSOC BGN Curncy", "JYSOD BGN Curncy",
        "JYSOE BGN Curncy", "JYSOF BGN Curncy", "JYSOG BGN Curncy", "JYSOH BGN Curncy",
        "JYSOI BGN Curncy", "JYSOJ BGN Curncy", "JYSOK BGN Curncy", "JYSO1 BGN Curncy",
        "JYSO1F BGN Curncy", "JYSO2 BGN Curncy", "JYSO3 BGN Curncy", "JYSO4 BGN Curncy",
        "JYSO5 BGN Curncy", "JYSO6 BGN Curncy", "JYSO7 BGN Curncy", "JYSO8 BGN Curncy",
        "JYSO9 BGN Curncy", "JYSO10 BGN Curncy", "JYSO12 BGN Curncy", "JYSO15 BGN Curncy",
        "JYSO20 BGN Curncy", "JYSO25 BGN Curncy", "JYSO30 BGN Curncy",
    ],
    "tnr": [
        1/12, 2/12, 3/12, 4/12, 5/12, 6/12,
        7/12, 8/12, 9/12, 10/12, 11/12, 1,
        1.5, 2, 3, 4, 5,
        6, 7, 8, 9, 10,
        12, 15, 20, 25, 30,
    ],
})

## Bloomberg data loader

In [ ]:
def get_bbg(tickers, field="PX_LAST", start_date="2000-01-01", end_date=None):
    """Thin wrapper around pxts.read_bdh for Bloomberg BDH requests."""
    return read_bdh(tickers, start=start_date, field=field, end=end_date)

## Curve builders

Three variants with increasing interpolation sophistication:

| Function | Interpolation | Notes |
|---|---|---|
| `get_curve` | Step / flat forward | Piecewise-constant between nodes |
| `get_curve_linear` | Linear forward | Piecewise-linear on inst. forward |
| `get_curve_cubic` | PCHIP cubic spline | Monotone cubic Hermite on A(t); **preferred** |

All three return a DataFrame of shape `(dates, 360)` with columns `m1 … m360`
representing annualised 1-month forward simple rates (% p.a.).

In [ ]:
def get_curve(ccy):
    """Step-constant (flat) forward bootstrap."""
    data = get_bbg(curve_tix[ccy].tolist(), start_date="2000-01-01")
    tnrs = curve_tix["tnr"].values

    frame = pd.DataFrame(
        np.nan, index=data.index, columns=[f"m{i}" for i in range(1, 361)]
    )

    for i in range(1, len(tnrs)):
        last_tnr = tnrs[i - 1]
        this_tnr = tnrs[i]
        fwd_tnr  = this_tnr - last_tnr

        last_rate = data.iloc[:, i - 1]
        this_rate = data.iloc[:, i]

        fwd_rate = 100 * (
            ((1 + this_rate / 100) ** this_tnr / (1 + last_rate / 100) ** last_tnr)
            ** (1 / fwd_tnr) - 1
        )

        cols = [f"m{j}" for j in range(round(last_tnr * 12) + 1, round(this_tnr * 12) + 1)]
        frame[cols] = np.column_stack([fwd_rate.values] * len(cols))

    frame["m1"] = data.iloc[:, 0].values
    return frame

In [ ]:
def get_curve_linear(ccy):
    """Piecewise-linear instantaneous-forward bootstrap."""
    data = get_bbg(curve_tix[ccy].tolist(), start_date="2000-01-01")
    tnrs = curve_tix["tnr"].values  # years

    # ── helpers ───────────────────────────────────────────────────────────────

    def _zero_to_discount(r, t, comp="cc"):
        if comp == "cc":     return np.exp(-r * t)
        if comp == "annual": return (1 + r) ** (-t)
        if comp == "simple": return 1 / (1 + r * t)

    def _A_at_t(t, t_nodes, f_nodes, A_nodes):
        """Cumulative area A(t) under piecewise-linear instantaneous forward."""
        t = np.asarray(t, dtype=float)
        # left-open interval: find i s.t. t_nodes[i] < t <= t_nodes[i+1]
        i = np.clip(
            np.searchsorted(t_nodes, t, side="left") - 1,
            0,
            len(t_nodes) - 2,
        )
        dt  = t - t_nodes[i]
        dti = t_nodes[i + 1] - t_nodes[i]
        incr = dt * (f_nodes[i] + 0.5 * (f_nodes[i + 1] - f_nodes[i]) * dt / dti)
        return A_nodes[i] + incr

    def _node_forwards(t_nodes, P_nodes):
        """Node instantaneous forwards consistent with given discount factors."""
        A = -np.log(P_nodes)
        n = len(t_nodes)
        f = np.zeros(n)
        r1 = A[0] / t_nodes[0]  # flat short end: f(0) = f(t1) = A1/t1
        f[0] = f[1] = r1
        for k in range(2, n):
            f[k] = 2 * (A[k] - A[k - 1]) / (t_nodes[k] - t_nodes[k - 1]) - f[k - 1]
        return f, A

    # ── main ──────────────────────────────────────────────────────────────────

    ord_idx = np.argsort(tnrs)
    t_nodes = tnrs[ord_idx]
    data_s  = data.iloc[:, ord_idx]

    t_grid = np.arange(361) / 12  # 0 … 30 years

    rows = []
    for irow in range(len(data_s)):
        zr = data_s.iloc[irow].values.astype(float)
        if np.isnan(zr).any():
            rows.append(np.full(360, np.nan))
            continue
        P = _zero_to_discount(zr / 100, t_nodes)
        f, A = _node_forwards(t_nodes, P)
        A_grid = np.empty(361)
        A_grid[0] = 0.0
        A_grid[1:] = _A_at_t(t_grid[1:], t_nodes, f, A)
        P_grid = np.exp(-A_grid)
        rows.append((P_grid[:-1] / P_grid[1:] - 1) * 12 * 100)

    return pd.DataFrame(
        np.vstack(rows), index=data.index, columns=[f"m{i}" for i in range(1, 361)]
    )

In [ ]:
def get_curve_cubic(ccy):
    """
    PCHIP cubic-spline bootstrap (preferred).

    Fits a monotone cubic Hermite spline (equivalent to R's monoH.FC) on
    A(t) = -log P(t) against a monthly time axis, then reads off 1-month
    simple forward rates for months 1 … 360.
    """
    data = get_bbg(curve_tix[ccy].tolist(), start_date="2000-01-01")
    tnrs = curve_tix["tnr"].values  # years

    # spline x-axis in months; sort ascending
    t_nodes_m = tnrs * 12
    ord_idx   = np.argsort(t_nodes_m)
    t_nodes_m = t_nodes_m[ord_idx]
    t_nodes_y = t_nodes_m / 12
    data_s    = data.iloc[:, ord_idx]

    t_grid_m = np.arange(361)  # 0 … 360 months
    delta_y  = 1 / 12

    rows = []
    for irow in range(len(data_s)):
        zr = data_s.iloc[irow].values.astype(float)
        if np.isnan(zr).any():
            rows.append(np.full(360, np.nan))
            continue
        P_nodes = np.exp(-zr / 100 * t_nodes_y)       # cc zero → discount
        A_nodes = -np.log(P_nodes)                     # cumulative area
        spline  = PchipInterpolator(t_nodes_m, A_nodes)
        A_grid  = spline(t_grid_m)
        A_grid[0] = 0.0                                # enforce A(0) = 0 exactly
        P_grid  = np.exp(-A_grid)
        F_1m    = (P_grid[:-1] / P_grid[1:] - 1) / delta_y * 100  # % p.a.
        rows.append(F_1m)

    return pd.DataFrame(
        np.vstack(rows), index=data.index, columns=[f"m{i}" for i in range(1, 361)]
    )

## Forward rate aggregation

`get_fwds(fwd_tnr, spt_tnr)` computes the annualised `spt_tnr`-month spot rate
starting `fwd_tnr` months from now, by chaining the 1-month forwards.

In [ ]:
def get_fwds(fwd_tnr, spt_tnr):
    """
    Annualised spot rate, spt_tnr months long, starting fwd_tnr months from now.

    Parameters
    ----------
    fwd_tnr : int   forward start in months
    spt_tnr : int   tenor in months

    Returns
    -------
    DataFrame (dates × CCYs)
    """
    parts = []
    for ccy in ccys:
        slct = [f"m{m}" for m in range(fwd_tnr + 1, fwd_tnr + spt_tnr + 1)]
        monthly_factors = (1 + curves[ccy][slct] / 100) ** (1 / 12)
        fwd = 100 * (monthly_factors.prod(axis=1) ** (12 / spt_tnr) - 1)
        fwd.name = ccy
        parts.append(fwd)
    return pd.concat(parts, axis=1)

## Plotting

In [ ]:
def plot_curves(**kwargs):
    """
    Plot 1-month forward curve cross-sections.

    Each keyword argument is  name=data  where:
      name : legend label (str)
      data : DataFrame row (1 × n) or Series with columns named 'm1' … 'mN'

    Example
    -------
    cols = [f"m{i}" for i in range(1, 61)]
    plot_curves(**{
        "EUR 2026-03-24": curves["EUR"].loc["2026-03-24", cols],
        "EUR 2026-03-17": curves["EUR"].loc["2026-03-17", cols],
    })
    """
    fig = go.Figure()

    for (name, data), color in zip(kwargs.items(), FT_COLORS):
        row = data.squeeze() if isinstance(data, pd.DataFrame) else data
        row = row.dropna()
        tenors = [int(c[1:]) / 12 for c in row.index]
        fig.add_trace(go.Scatter(
            x=tenors,
            y=row.values,
            mode="lines",
            name=name,
            line=dict(color=color, width=2),
        ))

    fig.update_layout(
        template="pxts",
        xaxis_title="Tenor (years)",
        yaxis_title="Forwards (%, 1-month parcels)",
        legend=dict(x=0.82, y=0.95, bgcolor="rgba(255,255,255,0.7)"),
    )
    return fig

## Load curves

Fetches Bloomberg data and bootstraps all four OIS forward curves.
This cell takes a minute or two on first run.

In [ ]:
ccys = ["EUR", "USD", "GBP", "JPY"]

curves = {ccy: get_curve_cubic(ccy) for ccy in ccys}

## Example: recent EUR curve cross-sections

In [ ]:
cols = [f"m{i}" for i in range(1, 61)]  # first 5 years

plot_curves(**{
    "EUR 2026-03-24": curves["EUR"].loc["2026-03-24", cols],
    "EUR 2026-03-17": curves["EUR"].loc["2026-03-17", cols],
    "EUR 2026-03-10": curves["EUR"].loc["2026-03-10", cols],
    "EUR 2026-03-03": curves["EUR"].loc["2026-03-03", cols],
    "EUR 2026-02-25": curves["EUR"].loc["2026-02-25", cols],
})